# Generador de nuevos Pokémon con una GAN (DCGAN)

Este notebook descarga sprites de Pokémon existentes (PokeAPI), entrena una DCGAN en PyTorch y genera Pokémon nuevos.

Dependencias: `torch`, `torchvision`, `numpy`, `pillow`, `matplotlib`, `tqdm`, `requests`. La celda de instalación las agrega con `pip`.

In [ ]:
# Instalación de dependencias (ejecutar una vez por entorno)
%pip install -q torch torchvision numpy pillow matplotlib tqdm requests

In [ ]:
import os, random, time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import requests
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision.utils import make_grid, save_image
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Dispositivo:", DEVICE)

# Configuración
ROOT = Path("." )
DATA_DIR = ROOT / "data" / "sprites"
OUT_DIR = ROOT / "output"
CKPT_DIR = ROOT / "checkpoints"
GEN_DIR = OUT_DIR / "generated"   # un PNG por cada Pokémon generado
for d in (DATA_DIR, OUT_DIR, CKPT_DIR, GEN_DIR): d.mkdir(parents=True, exist_ok=True)

# Estructura de assets (se crea sola):
#   data/sprites/      sprites descargados (se regeneran con el notebook)
#   output/            grillas de muestra por época y grilla final
#   output/generated/  Pokémon generados, un PNG por imagen
#   checkpoints/       pesos del modelo (last.pt)
(ROOT / ".gitignore").write_text(
    "# Sprites descargados (se regeneran con el notebook)\ndata/\n"
    "# Pesos del modelo\ncheckpoints/*.pt\n"
    "# Caches\n.ipynb_checkpoints/\n__pycache__/\n"
)

IMG_SIZE = 64
NZ = 100            # dimensión del vector latente
NGF = NDF = 64      # anchura base de las redes
BATCH_SIZE = 128
EPOCHS = 150
LR = 2e-4
N_POKEMON = 898     # ids 1..898 de PokeAPI
SAMPLE_EVERY = 10   # épocas entre grillas de muestra

## 1. Descarga de sprites
Se omiten los archivos ya descargados.

In [ ]:
URL = "https://raw.githubusercontent.com/PokeAPI/sprites/master/sprites/pokemon/{}.png"

def download(i):
    path = DATA_DIR / f"{i}.png"
    if path.exists() and path.stat().st_size > 0:
        return True
    try:
        r = requests.get(URL.format(i), timeout=20)
        if r.status_code == 200:
            path.write_bytes(r.content)
            return True
    except requests.RequestException:
        pass
    return False

with ThreadPoolExecutor(max_workers=16) as ex:
    results = list(tqdm(ex.map(download, range(1, N_POKEMON + 1)), total=N_POKEMON))
print(f"Sprites disponibles: {sum(results)} / {N_POKEMON}")

## 2. Dataset
Los PNG tienen transparencia: se componen sobre fondo blanco, se redimensionan a 64x64 y se normalizan a [-1, 1].

In [ ]:
class PokemonDataset(Dataset):
    def __init__(self, folder, size=IMG_SIZE, augment=True):
        self.files = sorted(Path(folder).glob("*.png"))
        self.size, self.augment = size, augment
        self.images = [self._load(f) for f in self.files]  # todo cabe en memoria

    def _load(self, f):
        img = Image.open(f).convert("RGBA")
        bg = Image.new("RGBA", img.size, (255, 255, 255, 255))
        img = Image.alpha_composite(bg, img).convert("RGB").resize((self.size, self.size), Image.LANCZOS)
        arr = np.asarray(img, dtype=np.float32) / 127.5 - 1.0
        return torch.from_numpy(arr).permute(2, 0, 1)

    def __len__(self): return len(self.images)

    def __getitem__(self, i):
        x = self.images[i]
        if self.augment and random.random() < 0.5:
            x = torch.flip(x, dims=[2])
        return x

dataset = PokemonDataset(DATA_DIR)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True,
                    num_workers=0, pin_memory=(DEVICE.type == "cuda"))
print("Imágenes:", len(dataset))

def show(batch, title="", nrow=8):
    g = make_grid(batch.cpu(), nrow=nrow, normalize=True, value_range=(-1, 1), padding=2)
    plt.figure(figsize=(8, 8)); plt.imshow(g.permute(1, 2, 0)); plt.axis("off"); plt.title(title); plt.show()

show(next(iter(loader))[:32], "Ejemplos reales")

## 3. Modelos

In [ ]:
class Generator(nn.Module):
    def __init__(self, nz=NZ, ngf=NGF):
        super().__init__()
        self.net = nn.Sequential(
            nn.ConvTranspose2d(nz, ngf * 8, 4, 1, 0, bias=False), nn.BatchNorm2d(ngf * 8), nn.ReLU(True),      # 4x4
            nn.ConvTranspose2d(ngf * 8, ngf * 4, 4, 2, 1, bias=False), nn.BatchNorm2d(ngf * 4), nn.ReLU(True),  # 8x8
            nn.ConvTranspose2d(ngf * 4, ngf * 2, 4, 2, 1, bias=False), nn.BatchNorm2d(ngf * 2), nn.ReLU(True),  # 16x16
            nn.ConvTranspose2d(ngf * 2, ngf, 4, 2, 1, bias=False), nn.BatchNorm2d(ngf), nn.ReLU(True),          # 32x32
            nn.ConvTranspose2d(ngf, 3, 4, 2, 1, bias=False), nn.Tanh(),                                         # 64x64
        )
    def forward(self, z): return self.net(z.view(z.size(0), -1, 1, 1))

class Discriminator(nn.Module):
    def __init__(self, ndf=NDF):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, ndf, 4, 2, 1, bias=False), nn.LeakyReLU(0.2, True),                                      # 32
            nn.Conv2d(ndf, ndf * 2, 4, 2, 1, bias=False), nn.BatchNorm2d(ndf * 2), nn.LeakyReLU(0.2, True),       # 16
            nn.Conv2d(ndf * 2, ndf * 4, 4, 2, 1, bias=False), nn.BatchNorm2d(ndf * 4), nn.LeakyReLU(0.2, True),   # 8
            nn.Conv2d(ndf * 4, ndf * 8, 4, 2, 1, bias=False), nn.BatchNorm2d(ndf * 8), nn.LeakyReLU(0.2, True),   # 4
            nn.Conv2d(ndf * 8, 1, 4, 1, 0, bias=False),                                                           # logit
        )
    def forward(self, x): return self.net(x).view(-1)

def init_weights(m):
    if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)):
        nn.init.normal_(m.weight, 0.0, 0.02)
    elif isinstance(m, nn.BatchNorm2d):
        nn.init.normal_(m.weight, 1.0, 0.02); nn.init.zeros_(m.bias)

G = Generator().to(DEVICE); D = Discriminator().to(DEVICE)
G.apply(init_weights); D.apply(init_weights)
print(f"Parámetros G: {sum(p.numel() for p in G.parameters()):,} | D: {sum(p.numel() for p in D.parameters()):,}")

## 4. Entrenamiento
Pérdida BCE con logits y label smoothing (reales = 0.9). Se guardan grillas con un ruido fijo para ver la evolución, y checkpoints.

In [ ]:
criterion = nn.BCEWithLogitsLoss()
optG = torch.optim.Adam(G.parameters(), lr=LR, betas=(0.5, 0.999))
optD = torch.optim.Adam(D.parameters(), lr=LR, betas=(0.5, 0.999))
fixed_noise = torch.randn(64, NZ, device=DEVICE)
REAL, FAKE = 0.9, 0.0
lossesG, lossesD = [], []

def train(epochs=EPOCHS, sample_every=SAMPLE_EVERY):
    for epoch in range(1, epochs + 1):
        G.train(); D.train()
        sG = sD = 0.0
        for real in tqdm(loader, desc=f"Época {epoch}/{epochs}", leave=False):
            real = real.to(DEVICE, non_blocking=True)
            b = real.size(0)
            z = torch.randn(b, NZ, device=DEVICE)
            fake = G(z)

            # Discriminador
            optD.zero_grad()
            lD = criterion(D(real), torch.full((b,), REAL, device=DEVICE)) + \
                 criterion(D(fake.detach()), torch.full((b,), FAKE, device=DEVICE))
            lD.backward(); optD.step()

            # Generador
            optG.zero_grad()
            lG = criterion(D(fake), torch.ones(b, device=DEVICE))
            lG.backward(); optG.step()
            sG += lG.item(); sD += lD.item()

        lossesG.append(sG / len(loader)); lossesD.append(sD / len(loader))
        print(f"Época {epoch:3d} | loss D {lossesD[-1]:.3f} | loss G {lossesG[-1]:.3f}")
        if epoch % sample_every == 0 or epoch == epochs:
            G.eval()
            with torch.no_grad(): samples = G(fixed_noise)
            save_image(samples, OUT_DIR / f"epoch_{epoch:03d}.png", nrow=8, normalize=True, value_range=(-1, 1))
            torch.save({"G": G.state_dict(), "D": D.state_dict(), "epoch": epoch}, CKPT_DIR / "last.pt")
            show(samples[:32], f"Época {epoch}")

train()

In [ ]:
plt.figure(figsize=(8, 3))
plt.plot(lossesG, label="Generador"); plt.plot(lossesD, label="Discriminador")
plt.xlabel("Época"); plt.ylabel("Pérdida"); plt.legend(); plt.show()

## 5. Generar nuevos Pokémon
Carga el último checkpoint y genera una grilla. Cambia `seed` para obtener otros resultados.

In [ ]:
def generate(n=32, seed=None, checkpoint=CKPT_DIR / "last.pt", save_as=None):
    ckpt = torch.load(checkpoint, map_location=DEVICE)
    G.load_state_dict(ckpt["G"]); G.eval()
    gen = torch.Generator(device=DEVICE)
    if seed is not None: gen.manual_seed(seed)
    z = torch.randn(n, NZ, device=DEVICE, generator=gen)
    with torch.no_grad(): imgs = G(z)
    show(imgs, f"Pokémon nuevos (época {ckpt['epoch']})")
    if save_as: save_image(imgs, save_as, nrow=8, normalize=True, value_range=(-1, 1))
    return imgs

new_pokemon = generate(32, seed=7, save_as=OUT_DIR / "nuevos_pokemon.png")

## 5b. Guardar los Pokémon generados como assets
Guarda cada imagen generada como un PNG individual en `output/generated/`, junto con la grilla en `output/`.

In [ ]:
def save_generated(imgs, folder=GEN_DIR, prefix="pokemon", scale=4):
    folder = Path(folder); folder.mkdir(parents=True, exist_ok=True)
    start = len(list(folder.glob(f"{prefix}_*.png")))
    for k, img in enumerate(imgs):
        arr = ((img.detach().cpu().permute(1, 2, 0).numpy() + 1) * 127.5).clip(0, 255).astype("uint8")
        im = Image.fromarray(arr).resize((IMG_SIZE * scale,) * 2, Image.NEAREST)
        im.save(folder / f"{prefix}_{start + k:04d}.png")
    print(f"Guardadas {len(imgs)} imágenes en {folder}")

save_generated(new_pokemon)

## 6. (Opcional) Interpolación en el espacio latente
Transición suave entre dos Pokémon generados.

In [ ]:
def interpolate(seed_a=1, seed_b=2, steps=10):
    G.eval()
    za = torch.randn(1, NZ, device=DEVICE, generator=torch.Generator(device=DEVICE).manual_seed(seed_a))
    zb = torch.randn(1, NZ, device=DEVICE, generator=torch.Generator(device=DEVICE).manual_seed(seed_b))
    ts = torch.linspace(0, 1, steps, device=DEVICE).view(-1, 1)
    with torch.no_grad(): imgs = G((1 - ts) * za + ts * zb)
    show(imgs, "Interpolación latente", nrow=steps)

interpolate(1, 2)